# 4. Performance

The final collection ran on Google Colab CPU across five runs. The tables below are generated from validated session telemetry and per-run benchmark evidence. Segment sequence and checksums were validated; the input hashes and revisions are recorded in `data/m1/final/provenance.json`. The two code revisions and overlapping execution prevent treating these observations as a controlled comparison.

The full-session sampler includes the notebook/runner process baseline and descendants. Its interval starts after initial setup/input hashing; preparation outside the sampler, including previously prepared S3 bulk acquisition, and final evidence export are not included. The recorded interval is therefore a **sampler wall**, not a claim of time for every external preparation step. Disk covers the configured local paths, not Drive storage; sampled peaks may miss short spikes.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, HTML

DATA = Path("../data/m1/final")
FIG = Path("../figures")
FIG.mkdir(exist_ok=True)
bench = pd.read_csv(DATA / "run_metrics.csv")
ts = pd.read_csv(DATA / "resource_samples_1min.csv")
attempts = pd.read_csv(DATA / "paper_attempt_timings.csv")
bench["start_utc"] = pd.to_datetime(bench.start_utc, utc=True)
bench["end_utc"] = pd.to_datetime(bench.end_utc, utc=True)

def display_table(frame):
    display(HTML('<div class="table-scroll">' + frame.to_html(index=False, na_rep="N/A") + "</div>"))


## 4.1. Per-run benchmark

Each row describes one measured session. `discovery_s` is the exported discovery diagnostic (including snapshot preparation where recorded); backoff is derived from available logs and may not include every reference retry. These components can overlap or already be included in session wall and must not be summed with it. The output sizes are session summaries, not a measurement of Drive quota.

The original timing audits for only-v1 and multi-Đăng reported incomplete session measurements. On a temporary copy of the exported evidence, rerunning the same revision (`ad03e0a3`) reproduced those results; placing the separately exported measurements under `work/measurements` made both audits complete. See [timing audit evidence](timing-audit-evidence.md) for the procedure and limits. Their chunk-only CSV `wall_time_e2e` labels are not used here. This does not establish the exact runtime state when the original audit ran or expand the sampler boundary.


In [ ]:
columns = ["run", "papers", "versions", "sampler_wall_s", "discovery_s", "retry_backoff_s", "peak_rss_mb", "mean_rss_mb", "peak_disk_mb", "output_mb"]
summary_table = bench[columns].copy()
for col in ("sampler_wall_s", "discovery_s", "retry_backoff_s"):
    summary_table[col] = pd.to_numeric(summary_table[col], errors="coerce").round().astype("Int64")
for col in ("peak_rss_mb", "mean_rss_mb", "peak_disk_mb", "output_mb"):
    summary_table[col] = summary_table[col].round(2)
display_table(summary_table)
display(HTML("<p class='table-caption'>Table 4.1. Per-run sampler time and resource observations; time is in seconds and memory and disk in MB.</p>"))
environment_table = bench[["run", "revision", "cpu_count", "machine_ram_mb", "start_utc", "end_utc", "original_timing_audit", "note"]].copy()
environment_table["machine_ram_mb"] = environment_table.machine_ram_mb.round(2)
for col in ("start_utc", "end_utc"):
    environment_table[col] = environment_table[col].dt.round("s")
display_table(environment_table)
display(HTML("<p class='table-caption'>Table 4.2. Code revision, machine configuration, session interval and original timing audit by run.</p>"))


## 4.2. RAM and disk over time

The plot uses approximately one observation per minute from each session and independent elapsed-time axes. The summary table uses the original sampler summaries, rather than recomputing peaks from this downsampled plot. Overlapping runtime memory and disk are not summed into a single machine's footprint.


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
for run, part in ts.groupby("run", sort=False):
    ax1.plot(part.t_sec / 3600, part.rss_mb, label=run)
    ax2.plot(part.t_sec / 3600, part.disk_mb, label=run)
ax1.set_ylabel("RSS (MB)")
ax2.set_ylabel("Local disk (MB)")
ax2.set_xlabel("Elapsed sampler time per session (hours)")
ax1.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "m1_resources.svg")


## 4.3. Per-paper attempt timing

An attempt's lifecycle runs from admission to recorded source/reference readiness and can include queue waiting. Attempts and papers overlap, so these durations are not additive wall time. The table covers recorded **completed attempts**, not necessarily one attempt per paper; retried papers can appear more than once.


In [ ]:
complete_attempts = attempts.loc[attempts.complete.astype(str).str.lower() == "true"].copy()
complete_attempts["lifecycle_s"] = pd.to_numeric(complete_attempts.lifecycle_s, errors="coerce")
per_attempt = complete_attempts.groupby("run").lifecycle_s.agg(["count", "mean", "median", "max"]).reset_index()
for col in ("mean", "median", "max"):
    per_attempt[col] = per_attempt[col].round().astype("Int64")
display_table(per_attempt.rename(columns={"mean": "mean_s", "median": "median_s", "max": "max_s"}))
display(HTML("<p class='table-caption'>Table 4.3. Completed attempt lifecycle durations by run, in seconds; attempts may overlap or repeat papers.</p>"))


## arXiv request timeline

`request_timeline.csv` has one row per minute of the crawl log: arXiv responses by status (`n_200`, `n_206` Range resumes, `n_429`, `n_5xx`), megabytes downloaded, seconds spent in backoff and the request pace in force. A throttled run shows `n_429` rising while the megabytes fall; a healthy one stays flat at the 3 s pace. The file comes with a run's exported evidence (spec 0004), so this section stays empty until the final CSVs are committed.

In [ ]:
timeline_path = DATA / 'request_timeline.csv'
if timeline_path.exists():
    tl = pd.read_csv(timeline_path)
    minutes = range(len(tl))
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
    ax1.bar(minutes, tl.n_429, color='tab:red', label='HTTP 429')
    ax1.bar(minutes, tl.n_5xx, bottom=tl.n_429, color='tab:orange', label='HTTP 5xx')
    ax1.set_ylabel('Refused requests / min')
    ax1.legend()
    ax2.plot(minutes, tl.mb_downloaded)
    ax2.set_ylabel('MB downloaded / min')
    ax2.set_xlabel('Minutes of the crawl log')
    fig.tight_layout()
    fig.savefig(FIG / 'm1_request_timeline.svg')
else:
    print('No request_timeline.csv in data/m1: placeholder data, nothing to plot.')


## 4.4. Concurrent runs and group elapsed interval

The table shows sampler start/end in UTC and the chart below places each session on a shared timeline. The envelope spans the earliest sampler start to the latest sampler end and **includes any gaps**; it excludes work outside the sampler boundaries. It is not the sum of session wall times, nor proof of a fully uninterrupted, end-to-end collection including earlier S3 downloads. Retry and checkpoint components must not be added again.


In [ ]:
timeline = bench.sort_values("start_utc")
timeline_table = timeline[["run", "start_utc", "end_utc", "sampler_wall_s"]].copy()
timeline_table["sampler_wall_s"] = timeline_table.sampler_wall_s.round().astype("Int64")
for col in ("start_utc", "end_utc"):
    timeline_table[col] = timeline_table[col].dt.round("s")
display_table(timeline_table)
display(HTML("<p class='table-caption'>Table 4.4. Sampler start and end in UTC, and sampler wall duration in seconds, by run.</p>"))
envelope_s = (bench.end_utc.max() - bench.start_utc.min()).total_seconds()
print(f"Sampler timeline envelope (includes gaps): {envelope_s / 3600:.2f} hours")
print(f"Sum of individual sampler walls (overlaps counted): {bench.sampler_wall_s.sum() / 3600:.2f} hours")
fig, ax = plt.subplots(figsize=(9, 3.5))
for _, row in timeline.iterrows():
    ax.barh(row["run"], (row.end_utc - row.start_utc).total_seconds() / 3600,
            left=(row.start_utc - bench.start_utc.min()).total_seconds() / 3600)
ax.invert_yaxis()
ax.set_xlabel("Hours since earliest sampler start (UTC)")
ax.set_title("Observed session intervals; gaps and overlaps shown")
fig.tight_layout()


## 4.5. Audit and limitations

The five manifests cover all assigned IDs with no overlap, and the expected version journal states were reconciled. This is not an independent readback of all raw corpus files. The original audit flags are preserved in the table; a controlled rerun explains how the exported measurement placement affects the results, without establishing the original runtime state. The two recorded code revisions, exclusion of earlier S3 preparation and changing service/network conditions limit comparisons between runs.


## 4.6. arXiv HTTP 429 and recorded backoff over time

Each point is an observed minute in a run's exported request timeline, plotted in UTC; minutes with zero events are omitted from the scatter. The upper panel counts logged arXiv 429 responses. The lower panel aggregates waits matching the exporter's backoff log patterns; it does not by itself attribute every wait to a specific service. Logs and parser scope may omit events or reference retries, and absence of a point is not proof that no request occurred. Backoff is already included in the relevant running intervals and must not be added to sampler wall.


In [ ]:
timeline_requests = pd.read_csv(DATA / "request_timeline_by_run.csv")
timeline_requests["minute_utc"] = pd.to_datetime(timeline_requests.minute_utc, utc=True)
fig, (ax429, axwait) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for run, part in timeline_requests.groupby("run", sort=False):
    observed_429 = part.loc[part.n_429 > 0]
    observed_wait = part.loc[part.backoff_s > 0]
    ax429.scatter(observed_429.minute_utc, observed_429.n_429, label=run, s=14, alpha=.75)
    axwait.scatter(observed_wait.minute_utc, observed_wait.backoff_s, label=run, s=14, alpha=.75)
ax429.set_ylabel("arXiv 429 / minute")
axwait.set_ylabel("Recorded backoff (s / minute)")
axwait.set_xlabel("UTC")
ax429.legend(fontsize=8, ncol=2)
fig.autofmt_xdate()
fig.tight_layout()
request_summary = timeline_requests.groupby("run")[["n_429", "n_5xx", "backoff_s"]].sum()
request_summary = request_summary.reset_index()
request_summary["backoff_s"] = request_summary.backoff_s.round().astype("Int64")
display_table(request_summary)
display(HTML("<p class='table-caption'>Table 4.5. Logged arXiv HTTP 429 and 5xx responses and recorded backoff seconds by run.</p>"))


## 4.7. S3 bulk download benchmark and estimated end-to-end time

The team downloaded the bulk previously used for the M1 collection without an audited download benchmark. The results below come from a **separate repeat download on 8 November 2026**, after the crawl sessions, using the same assigned 20,000-ID snapshot and a recorded manifest. They do not establish the duration or speed of the original download. The manifest ranges identify candidate bundles; they do not establish that each paper or version is present.

The download session starts after input selection and AWS setup. `selection_s` is reported separately. Download, local MD5 verification and copy to Drive are sequential bundle timings within session wall; the remainder covers unclassified time. Local MD5 validation precedes copying. Closing or renaming files through Drive FUSE does not independently prove durable cloud upload; the evidence includes no independent full Drive readback or member-level coverage audit. All 144 sizes differ from the manifest even though the local MD5 checks succeeded. Resource peaks are sampled every two seconds for each bundle: RSS covers the process tree and disk covers only the local staging directory, so short peaks may be missed.


In [ ]:
import json

s3 = json.loads((DATA / "s3_download_summary.json").read_text(encoding="utf-8"))
remaining = s3["session_wall_s"] - s3["download_s"] - s3["verify_s"] - s3["copy_s"]
crawl_envelope_s = (bench.end_utc.max() - bench.start_utc.min()).total_seconds()
estimated_e2e_s = s3["session_wall_s"] + crawl_envelope_s
s3_table = pd.DataFrame([
    ("Assigned IDs", f"{s3['papers']:,}"),
    ("Downloaded / reused / failed bundles", f"{s3['downloaded']} / {s3['reused']} / {s3['failed']}"),
    ("Actual / manifest size (GB)", f"{s3['actual_bytes'] / 1e9:.2f} / {s3['manifest_bytes'] / 1e9:.2f}"),
    ("Download session wall (s)", round(s3["session_wall_s"])),
    ("Selection outside session (s)", round(s3["selection_s"])),
    ("Download / local MD5 / Drive copy (s)", f"{round(s3['download_s'])} / {round(s3['verify_s'])} / {round(s3['copy_s'])}"),
    ("Download / session throughput (MB/s)", f"{s3['actual_bytes'] / s3['download_s'] / 1e6:.2f} / {s3['actual_bytes'] / s3['session_wall_s'] / 1e6:.2f}"),
    ("Peak process-tree RSS / local staging (MiB)", f"{s3['peak_rss_bytes'] / 2**20:.1f} / {s3['peak_stage_bytes'] / 2**20:.1f}"),
    ("Crawl sampler envelope including gaps (s)", round(crawl_envelope_s)),
    ("Estimated download plus crawl (s)", round(estimated_e2e_s)),
], columns=["Metric", "Value"])
display_table(s3_table)
display(HTML("<p class='table-caption'>Table 4.5. Repeat S3 download benchmark and estimated end-to-end duration. Seconds are rounded for display.</p>"))
fig, ax = plt.subplots(figsize=(10, 2.2))
left = 0
for label, seconds, color in [("Download", s3["download_s"], "#337d79"),
                              ("Local MD5", s3["verify_s"], "#e5ad43"),
                              ("Drive copy", s3["copy_s"], "#bf6659"),
                              ("Other", remaining, "#888888")]:
    ax.barh(["S3 session"], [seconds], left=left, label=f"{label}: {round(seconds)} s", color=color)
    left += seconds
ax.set_xlabel("Seconds within download session")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.35), ncol=2)
fig.tight_layout()


The end-to-end figure is an **estimate assembled from two measurements**: the repeat S3 download session plus the interval from the earliest crawl sampler start to the latest end. The crawl interval includes gaps and already accounts for overlapping sessions; individual session durations, retry waits and checkpoint times are not added again. Input selection, mounting Drive, setup, evidence export and other work outside the recorded intervals remain excluded. This is not an observed continuous wall-clock duration of the original M1 run. Input hashes and the download code revision are recorded in `data/m1/final/s3_download_provenance.json`.
